In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/tastelet/data")
RAW_DIR = DATA_DIR / "raw"
MOVIELENS_DIR = RAW_DIR / "ml-latest-small"

if not MOVIELENS_DIR.exists():
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    !wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip -O /content/ml.zip
    !unzip -q /content/ml.zip -d "{RAW_DIR}"

print(sorted(p.name for p in MOVIELENS_DIR.iterdir()))

['README.txt', 'links.csv', 'movies.csv', 'ratings.csv', 'tags.csv']


In [5]:
import pandas as pd

ratings = pd.read_csv(MOVIELENS_DIR / "ratings.csv")
print(ratings.shape)

(100836, 4)


In [6]:
movies = pd.read_csv(MOVIELENS_DIR / "movies.csv")
links = pd.read_csv(MOVIELENS_DIR / "links.csv")
print(movies.shape, links.shape)

(9742, 3) (9742, 3)


In [7]:
from google.colab import userdata

TMDB_API_KEY = userdata.get("TMDB_API_KEY")
print(len(TMDB_API_KEY))

32


In [8]:
print(links["tmdbId"].isna().sum())

8


In [9]:
links = links.dropna(subset=["tmdbId"]).astype({"tmdbId": int})
print(links.shape)

(9734, 3)


In [10]:
rating_counts = ratings["movieId"].value_counts()
print((rating_counts < 5).sum())

6074


In [11]:
MIN_RATINGS_PER_MOVIE = 5

kept_movie_ids = rating_counts[rating_counts >= MIN_RATINGS_PER_MOVIE].index.intersection(links["movieId"])
ratings = ratings[ratings["movieId"].isin(kept_movie_ids)]
print(len(kept_movie_ids), ratings.shape)

3650 (90274, 4)


In [12]:
movies = movies[movies["movieId"].isin(kept_movie_ids)]
print(movies.shape)

(3650, 3)


In [13]:
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)\s*$", expand=False)
print(movies["year"].isna().sum())

0


In [15]:
movies["year"] = movies["year"].astype(int)
print(movies.head(3))

   movieId                    title  \
0        1         Toy Story (1995)   
1        2           Jumanji (1995)   
2        3  Grumpier Old Men (1995)   

                                        genres  year  
0  Adventure|Animation|Children|Comedy|Fantasy  1995  
1                   Adventure|Children|Fantasy  1995  
2                               Comedy|Romance  1995  


In [16]:
import requests

TMDB_MOVIE_URL = "https://api.themoviedb.org/3/movie/{tmdb_id}"

response = requests.get(TMDB_MOVIE_URL.format(tmdb_id=862), params={"api_key": TMDB_API_KEY}, timeout=10)
print(response.status_code)
print(response.json().get("title"), response.json().get("poster_path"))

200
Toy Story /uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg


In [17]:
def fetch_tmdb_movie(tmdb_id: int) -> dict | None:
    response = requests.get(TMDB_MOVIE_URL.format(tmdb_id=tmdb_id), params={"api_key": TMDB_API_KEY}, timeout=10)
    if response.status_code == 404:
        return None
    response.raise_for_status()
    movie = response.json()
    return {"poster_path": movie.get("poster_path"), "overview": movie.get("overview")}

print(fetch_tmdb_movie(862))
print(fetch_tmdb_movie(999999999))

{'poster_path': '/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg', 'overview': "Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences."}
None


In [18]:
import json

TMDB_CACHE_PATH = RAW_DIR / "tmdb_cache.json"

tmdb_cache = json.loads(TMDB_CACHE_PATH.read_text()) if TMDB_CACHE_PATH.exists() else {}
tmdb_ids_to_fetch = [tmdb_id for tmdb_id in links.loc[links["movieId"].isin(kept_movie_ids), "tmdbId"] if str(tmdb_id) not in tmdb_cache]
print(len(tmdb_cache), len(tmdb_ids_to_fetch))

0 3650


In [19]:
import time

TMDB_SLEEP_SECONDS = 0.05
CACHE_SAVE_EVERY = 100

for i, tmdb_id in enumerate(tmdb_ids_to_fetch, start=1):
    tmdb_cache[str(tmdb_id)] = fetch_tmdb_movie(tmdb_id)
    time.sleep(TMDB_SLEEP_SECONDS)
    if i % CACHE_SAVE_EVERY == 0:
        TMDB_CACHE_PATH.write_text(json.dumps(tmdb_cache))
        print(i, "fetched")

TMDB_CACHE_PATH.write_text(json.dumps(tmdb_cache))
print("done:", len(tmdb_cache))

100 fetched
200 fetched
300 fetched
400 fetched
500 fetched
600 fetched
700 fetched
800 fetched
900 fetched
1000 fetched
1100 fetched
1200 fetched
1300 fetched
1400 fetched
1500 fetched
1600 fetched
1700 fetched
1800 fetched
1900 fetched
2000 fetched
2100 fetched
2200 fetched
2300 fetched
2400 fetched
2500 fetched
2600 fetched
2700 fetched
2800 fetched
2900 fetched
3000 fetched
3100 fetched
3200 fetched
3300 fetched
3400 fetched
3500 fetched
3600 fetched
done: 3650


In [20]:
tmdb_ids_without_poster = [tmdb_id for tmdb_id, meta in tmdb_cache.items() if meta is None or not meta["poster_path"]]
print(len(tmdb_ids_without_poster))

14


In [21]:
movie_ids_without_poster = links.loc[links["tmdbId"].astype(str).isin(tmdb_ids_without_poster), "movieId"]

movies = movies[~movies["movieId"].isin(movie_ids_without_poster)]
ratings = ratings[~ratings["movieId"].isin(movie_ids_without_poster)]
print(movies.shape, ratings.shape)

(3636, 4) (90111, 4)
